# Cross-target CMI — Visual companion

Visual reading of the cross-target CMI experiment on Lending Club.
Five sections, one chart each, each making one point:

1. The joint structure of $(Y^{(1)}, Y^{(2)})$ — why the two targets aren't redundant.
2. ROC overlay — what AUC sees on each target.
3. Score distributions stratified by $Y^{(1)}$ — what AUC misses.
4. The CMI triad with error bars — the formal portability measurement.
5. AUC vs normalized CMI side by side — the two metrics, contrasted.

Reuses estimator functions from `cross_target_pipeline.py` in this folder.
Run end-to-end to reproduce the post's figures.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

sys.path.insert(0, str(Path.cwd()))
from cross_target_pipeline import (
    load_cohort, build_xy, oof_score,
    cf_marginal_cmi, cf_conditional_cmi,
    conditional_entropy_Y2_given_Y1,
)

# Plot style — keep it minimal.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

PAID = "#3b82c4"     # Y=0
LOSS = "#e88c30"     # Y=1
NEUTRAL = "#8a8a8a"

SAMPLE = 100_000   # plenty for visuals; pipeline already validated at 200K
SEED = 0


## Data: cohort + features + out-of-fold score

Reuses the pipeline's cached cohort (36-month term, terminal status) and feature
encoding. Subsamples to 100k for speed; the chart conclusions are stable from ~50k up.


In [ ]:
df = load_cohort(SAMPLE, SEED)
X, Y1, Y2, cat_idx = build_xy(df)
del df

print(f"n = {len(Y1):,}")
print(f"p(Y(1) = 1) = {Y1.mean():.4%}   (late fee accrued)")
print(f"p(Y(2) = 1) = {Y2.mean():.4%}   (terminal loss)")


In [ ]:
# Out-of-fold S^{(1)} — the cross-target score, trained on Y^{(1)}.
# This is the score whose portability we're measuring.
S = oof_score(X, Y1, cat_idx, K=5, seed=SEED)
auc_S1_Y1 = roc_auc_score(Y1, S)
auc_S1_Y2 = roc_auc_score(Y2, S)
print(f"AUC(S^(1), Y^(1)) = {auc_S1_Y1:.4f}")
print(f"AUC(S^(1), Y^(2)) = {auc_S1_Y2:.4f}")


In [ ]:
# Out-of-fold S^{(2)} — the baseline. Same architecture, same features,
# trained directly on Y^{(2)}.  This is the natural reference for "what
# would retraining buy us?" — the upper bound on conditional information
# extractable by this model class with these features.
S2 = oof_score(X, Y2, cat_idx, K=5, seed=SEED)
auc_S2_Y2 = roc_auc_score(Y2, S2)
print(f"AUC(S^(2), Y^(2)) = {auc_S2_Y2:.4f}")


## 1. The joint structure of $(Y^{(1)}, Y^{(2)})$

Neither target contains the other. There are loans with no late fee that still
charged off (top right of the matrix below), and loans that accrued a late fee
and were paid in full (bottom left). The off-diagonal cells are what make the
cross-target question non-trivial — if either contained the other, the CMI
would degenerate.


In [ ]:
# 2x2 joint distribution
counts = pd.crosstab(pd.Series(Y1, name="Y(1)"),
                     pd.Series(Y2, name="Y(2)"))
pcts = counts / counts.values.sum() * 100

fig, ax = plt.subplots(figsize=(5.2, 4.0))
im = ax.imshow(pcts.values, cmap="Blues", vmin=0, aspect="auto")
for i in range(2):
    for j in range(2):
        v = pcts.iloc[i, j]
        n = counts.iloc[i, j]
        ax.text(j, i, f"{v:.1f}%\n(n={n:,})",
                ha="center", va="center", fontsize=11,
                color="white" if v > 40 else "black")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Y(2)=0\n(paid)", "Y(2)=1\n(loss)"])
ax.set_yticklabels(["Y(1)=0\n(no late fee)", "Y(1)=1\n(late fee)"])
ax.set_title("Joint distribution of (Y(1), Y(2))")
plt.colorbar(im, ax=ax, label="% of cohort", fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 2. ROC overlay — what AUC sees

The same score $S^{(1)}$ — trained out-of-fold to predict $Y^{(1)}$ — evaluated
on both targets. The two ROC curves are essentially on top of each other; the
AUCs differ by less than half a percentage point. From AUC alone you'd conclude
the score is *about equally useful* for either target.

That conclusion is correct as far as it goes — but it's not the whole picture.


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for Y, name, color in [(Y1, r"$Y^{(1)}$  (late fee)", PAID),
                       (Y2, r"$Y^{(2)}$  (loss)",      LOSS)]:
    fpr, tpr, _ = roc_curve(Y, S)
    auc = roc_auc_score(Y, S)
    ax.plot(fpr, tpr, label=f"{name}   AUC = {auc:.3f}",
            color=color, lw=2)
ax.plot([0, 1], [0, 1], "--", color=NEUTRAL, lw=1, alpha=0.7,
        label="chance")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("Same score, two targets")
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 3. Within each $Y^{(1)}$ stratum, does the score still separate $Y^{(2)}$?

This is the visual that AUC can't deliver. Within each value of $Y^{(1)}$,
plot the score distribution and color by $Y^{(2)}$. If the orange (loss)
distribution sits to the right of the blue (paid) distribution *within each
stratum*, the score is doing real conditional work — discriminating $Y^{(2)}$
beyond what knowing $Y^{(1)}$ already tells you.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
fig.suptitle("Within each Y(1) stratum, does the score still separate Y(2)?",
             y=1.00, fontsize=12)

for ax, y1_val, subtitle in [(axes[0], 0, "no late fee"),
                              (axes[1], 1, "late fee accrued")]:
    mask = (Y1 == y1_val)
    S_str = S[mask]; Y2_str = Y2[mask]
    s_max = np.percentile(S_str, 99.5)  # clip tail for readable scale
    bins = np.linspace(0, s_max, 35)

    ax.hist(S_str[Y2_str == 0], bins=bins, density=True,
            alpha=0.55, color=PAID, label="Y(2) = 0 (paid)")
    ax.hist(S_str[Y2_str == 1], bins=bins, density=True,
            alpha=0.55, color=LOSS, label="Y(2) = 1 (loss)")

    # Median lines make the conditional shift visible despite heavy overlap.
    m0 = np.median(S_str[Y2_str == 0])
    m1 = np.median(S_str[Y2_str == 1])
    ax.axvline(m0, color=PAID, lw=2, ls="--", alpha=0.9)
    ax.axvline(m1, color=LOSS, lw=2, ls="--", alpha=0.9)

    p_y2 = Y2_str.mean()
    ax.set_title(f"Y(1) = {y1_val}  ({subtitle})\n"
                 f"n = {mask.sum():,},  P(Y(2)=1) = {p_y2:.1%},  "
                 f"median shift = {(m1 - m0):+.4f}",
                 fontsize=10)
    ax.set_xlabel(r"$S^{(1)}$")
    ax.set_ylabel("density")
    ax.legend(frameon=False, loc="upper right")
    ax.set_xlim(0, s_max)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## 4. The CMI triad

Three numbers — the score's self-information, its marginal information about
the new target, and its conditional information about the new target given
$Y^{(1)}$ — with fold-level standard errors as 95% intervals.

The headline observation: $I(S; Y^{(2)} \mid Y^{(1)})$ preserves the lion's
share of $I(S; Y^{(2)})$. The score's predictive content for $Y^{(2)}$ is
mostly *not* mediated by $Y^{(1)}$; it's X-level structure the model picked up
while training to predict late fees, that turns out to predict charge-off too.


In [ ]:
# S^{(1)} triad
si, si_se = cf_marginal_cmi(S,  Y1, K=5, seed=SEED)
mc, mc_se = cf_marginal_cmi(S,  Y2, K=5, seed=SEED)
cc, cc_se = cf_conditional_cmi(S, Y1, Y2, K=5, seed=SEED)

# S^{(2)} baseline (same conditional structure, different score)
cc2, cc2_se = cf_conditional_cmi(S2, Y1, Y2, K=5, seed=SEED)

h_y2_y1     = conditional_entropy_Y2_given_Y1(Y1, Y2)
ratio_S1    = cc  / h_y2_y1
ratio_S2    = cc2 / h_y2_y1
portability = cc / cc2

print(f"I(S^(1); Y^(1))            = {si:.4f}  +/- {si_se:.4f}")
print(f"I(S^(1); Y^(2))            = {mc:.4f}  +/- {mc_se:.4f}")
print(f"I(S^(1); Y^(2) | Y^(1))    = {cc:.4f}  +/- {cc_se:.4f}")
print(f"I(S^(2); Y^(2) | Y^(1))    = {cc2:.4f}  +/- {cc2_se:.4f}    [baseline]")
print(f"H(Y^(2) | Y^(1))           = {h_y2_y1:.4f}")
print(f"")
print(f"S^(1) resolves   {ratio_S1:.2%}  of remaining Y^(2)-uncertainty | Y^(1)")
print(f"S^(2) resolves   {ratio_S2:.2%}  (same-architecture baseline)")
print(f"portability  =   {portability:.2%}  (S^(1) / S^(2))")


In [ ]:
labels = [r"$I(S; Y^{(1)})$",
          r"$I(S; Y^{(2)})$",
          r"$I(S; Y^{(2)} \mid Y^{(1)})$"]
means = [si, mc, cc]
ses   = [si_se, mc_se, cc_se]
colors = [PAID, LOSS, "#5b8e3a"]

fig, ax = plt.subplots(figsize=(7.5, 3.2))
y_pos = np.arange(len(labels))
ax.errorbar(means, y_pos, xerr=[1.96 * s for s in ses],
            fmt="o", markersize=8, capsize=4, color="black",
            ecolor="gray", elinewidth=1.4)
for y, m, c in zip(y_pos, means, colors):
    ax.plot(m, y, "o", color=c, markersize=8, zorder=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel("nats")
ax.set_xlim(0, max(means) * 1.35)
ax.invert_yaxis()
ax.axvline(0, color=NEUTRAL, lw=0.8, alpha=0.4)
ax.grid(axis="x", alpha=0.3)
ax.set_title("CMI triad   (point ± 95% from fold SE)")

ax.text(0.98, 0.96,
        f"conditional / marginal = {cc/mc:.0%}\n"
        "(descriptive ratio, not chain-rule)",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, color="dimgray",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="lightgray"))
plt.tight_layout()
plt.show()


## 5. Cross-target portability against the retraining baseline

The interesting comparison isn't "score vs nothing" — it's *the cross-target
score vs a same-architecture model trained directly on $Y^{(2)}$*. That's the
practitioner-relevant question: would I gain by retraining? The baseline
$S^{(2)}$ uses the same features, same model class, same fold structure;
the only difference is which target it was fit on.

Left panel: AUC on $Y^{(2)}$ for both scores. The retrained $S^{(2)}$ beats
the cross-target $S^{(1)}$ by ~4 AUC points — a real, but not dramatic, gap.

Right panel: the fraction of remaining $Y^{(2)}$-uncertainty (given $Y^{(1)}$)
that each score resolves. $S^{(1)}$ resolves about 3.7%; $S^{(2)}$ resolves
about 6.7%. The cross-target score captures roughly half of what retraining
would deliver. *That ratio is the decision-relevant portability scalar*.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8),
                         gridspec_kw={"width_ratios": [0.9, 1.4]})

S1_COLOR = "#3b82c4"   # cross-target
S2_COLOR = "#5b8e3a"   # retrained baseline

# Left: AUC on Y^{(2)} — cross-target vs retrained
ax = axes[0]
bars = ax.bar([r"$S^{(1)}$ (cross-target)", r"$S^{(2)}$ (retrained)"],
              [auc_S1_Y2, auc_S2_Y2],
              color=[S1_COLOR, S2_COLOR], edgecolor="white", linewidth=2)
for b, v in zip(bars, [auc_S1_Y2, auc_S2_Y2]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.006, f"{v:.3f}",
            ha="center", fontsize=11)
gap = auc_S2_Y2 - auc_S1_Y2
ax.annotate("", xy=(1, auc_S2_Y2 - 0.005), xytext=(1, auc_S1_Y2 + 0.005),
            arrowprops=dict(arrowstyle="<->", color="gray", lw=1.2))
ax.text(1.15, (auc_S1_Y2 + auc_S2_Y2)/2, f"+{gap:.3f}",
        ha="left", va="center", fontsize=10, color="gray")
ax.axhline(0.5, color=NEUTRAL, ls="--", lw=1, alpha=0.5)
ax.set_ylim(0.5, max(auc_S1_Y2, auc_S2_Y2) + 0.04)
ax.set_ylabel("AUC on $Y^{(2)}$")
ax.set_title("AUC view")
ax.grid(alpha=0.3, axis="y")

# Right: conditional-CMI resolution — two stacked horizontal bars
ax = axes[1]
ratio_S1_pct = ratio_S1 * 100
ratio_S2_pct = ratio_S2 * 100

ax.barh([1], [ratio_S1_pct], color=S1_COLOR, height=0.55,
        label=f"$S^{{(1)}}$ (cross-target): {ratio_S1_pct:.1f}%")
ax.barh([1], [100 - ratio_S1_pct], left=[ratio_S1_pct],
        color="lightgray", height=0.55)
ax.barh([0], [ratio_S2_pct], color=S2_COLOR, height=0.55,
        label=f"$S^{{(2)}}$ (retrained):  {ratio_S2_pct:.1f}%")
ax.barh([0], [100 - ratio_S2_pct], left=[ratio_S2_pct],
        color="lightgray", height=0.55)

# Light dashed leader showing the gap
ax.plot([ratio_S1_pct, ratio_S1_pct], [0.275, 1 - 0.275],
        color="gray", ls=":", lw=1.0)
ax.plot([ratio_S2_pct, ratio_S2_pct], [0 - 0.275, 0.275],
        color="gray", ls=":", lw=1.0)
ax.text((ratio_S1_pct + ratio_S2_pct)/2, 0.5,
        f"retraining gain\n+{(ratio_S2_pct - ratio_S1_pct):.1f} pp",
        ha="center", va="center", fontsize=9, color="gray",
        bbox=dict(boxstyle="round,pad=0.3", fc="white",
                  ec="lightgray", alpha=0.95))

ax.set_yticks([0, 1])
ax.set_yticklabels([r"$S^{(2)}$", r"$S^{(1)}$"])
ax.set_xlim(0, 100)
ax.set_ylim(-0.6, 1.6)
ax.set_xlabel(r"% of $H(Y^{(2)} \mid Y^{(1)}) = $" + f"{h_y2_y1:.3f} nats")
ax.set_title(f"Conditional-information view  —  "
             f"portability = {portability:.0%}")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20),
          ncol=2, frameon=False, fontsize=9)

plt.tight_layout()
plt.show()


## Reading the result

Two facts hold simultaneously, and the post needs to land both:

**Structurally, the score is portable.** Section 4: the conditional CMI
$I(S^{(1)}; Y^{(2)} \mid Y^{(1)})$ has magnitude about 84% of the marginal
$I(S^{(1)}; Y^{(2)})$. (That's a descriptive ratio, not a chain-rule
decomposition — the chain rule decomposes the joint MI of the *pair*
$(S^{(1)}, Y^{(1)})$, not the marginal MI of $S^{(1)}$ alone.) The
conditional content doesn't shrink much when $Y^{(1)}$ enters the
conditioning set, which is evidence that the score's predictive content
for $Y^{(2)}$ is mostly genuine X-level structure rather than information
inherited from the $Y^{(1)} \to Y^{(2)}$ label overlap. A model trained on
the noisier proxy target (late-fee accrual at 4% prevalence) ends up
encoding the underlying credit-risk signal carried by the application-time
features.

**Operationally, retraining leaves real conditional information on the table.**
Section 5: a same-architecture, same-features model trained directly on
$Y^{(2)}$ resolves about 6.7% of the remaining $Y^{(2)}$-uncertainty given
$Y^{(1)}$, versus 3.7% for the cross-target score. The cross-target score
captures about 56% of what retraining delivers in this model class —
*partial portability*, not *full*. Whether the 44% gap is worth retraining
for depends on the operational picture the CMI alone doesn't tell you:
deployment friction, audit requirements, downstream consumers of the
existing score. The properly decision-relevant information-theoretic
quantity is the *unique score contribution*
$I(S^{(2)}; Y^{(2)} \mid S^{(1)})$ — does the retrained score add anything
beyond the existing one? — which is the headline of the next post in this
series.

The AUC and CMI views agree in *direction* (both say retraining helps) but
disagree in *magnitude*. AUC reports a 4-point gap (0.65 → 0.69), which a
practitioner might read as small. The normalized CMI reports a near-doubling
(3.7% → 6.7%) — the same gap, but expressed as a fraction of resolvable
uncertainty rather than rank discrimination. Both are correct; reporting
only one risks miscalibration of the retraining decision.

### Two caveats worth naming explicitly

**Policy-dependent features.** Some of the application-time features —
`int_rate`, `grade`, `sub_grade` in particular — encode Lending Club's
own internal risk scoring at origination. A score trained on these features
is partly imitating LC's grading model, not just learning the underlying
credit-risk structure. The conditional CMI we measured is a valid
portability diagnostic *for the score we trained*, but the score's
portability inherits whatever LC's grading model encodes. A cleaner
experiment would drop these features and rerun; the qualitative story is
unlikely to flip, but the absolute numbers would shift.

**Calibration of the conditional auxiliary.** The log-loss-gain CMI is
calibration-sensitive: it is only an unbiased estimator of conditional MI
when the auxiliary $\hat{p}_1(Y^{(2)} \mid S, Y^{(1)})$ is well-calibrated.
HistGBDT outputs are not strongly calibrated by default. If both
auxiliaries are miscalibrated symmetrically, the bias cancels in the
difference; if asymmetrically, it does not. The natural robustness check
is to refit the full auxiliary as an isotonic regression stratified by
$Y^{(1)}$, which is the cheapest flexible calibrator, and verify the CMI
estimate is close to the GBDT version. If the two agree, calibration isn't
doing the heavy lifting.
